# Multi-step Reasoning and Planning


> Earlier lectures each ease a limit of single-step reasoning: Lecture 2 spends compute at inference time through repeated sampling, Lecture 3 scores a solution with a verifier, and Lecture 4 lets the model alternate with the environment and receive tool feedback. All of those methods still advance along a single path: if the current step is wrong, there is no way back.
>
> This lecture treats multi-step work as a planning problem: how many steps to split into, how many candidates to keep, which steps can run in parallel, and whether each step should widen or deepen. We take ideas from four papers — ADaPT, LATS, SPRINT, and Wider or Deeper — and implement their core loops from scratch in numpy.

Start with a minimal example. Give the model a goal: craft a wooden sword.

Step one, look up the recipe. A wooden sword needs two ingredients — a plank and an iron ingot — and both must be on hand before the craft can succeed.

Step two, notice that a plank is not atomic. The plank recipe is a stick, so a stick must exist before a plank can be made. The iron ingot is atomic and can be used as is.

Step three, order the work: make the stick first, then craft the plank from the stick, then combine the plank with the iron ingot to make the sword. The order is stick → plank → (with iron) → sword.

If the model starts crafting the sword without this dependency chain, it will stall halfway when the plank is still missing. Deciding which subgoals to complete, and in which order, before acting is `planning`.

The ReAct loop from Lecture 4 already lets the model act step by step, but each step follows only the currently best direction; if any intermediate choice is wrong, recovery is hard. The hard part of a multi-step task is often not the individual action, but the choice made before acting: whether to split the task, into how many pieces, and which piece to do first. Two common responses match two kinds of difficulty: splitting a large task into subgoals and completing them one by one is `task decomposition`; keeping several candidate routes and switching when one fails is `tree search`.

By the end of this lecture we will have implemented an Agent loop that plans before it acts. Placed in the perceive–decide–act–feedback cycle from Lecture 1, this lecture strengthens `decision`: instead of picking one action at each step, the Agent plans first and then acts. We start from where single-step reasoning gets stuck.

This section uses a small example to show where single-step reasoning gets stuck. Decomposition and search later in the lecture are responses to that failure.

Two naive ways to handle a multi-step task are to let the model emit a full answer in one shot (holistic generation), or to walk forward one step at a time (greedy decoding). Holistic generation is unreliable on long tasks: an error in the middle pulls the ending with it. Greedy decoding keeps only the currently best choice at each step; once that choice is wrong, earlier states are gone.

A numeric chain makes the point. Start at 1, apply three operations, and aim for 40. The correct steps are ×4, +6, ×4. If the model outputs +2 at step 2, the path ends at 24; even if later steps stay correct, 24 cannot be pulled back to 40. The wrong value has already entered the later arithmetic, and a single path does not keep other possibilities.

In [ ]:
# One greedy single-step path: start at 1, apply 3 operations, target 40.
# The correct steps are [×4, +6, ×4]; step 2 will be replaced by +2.
true_steps = [("×", 4), ("+", 6), ("×", 4)]
bad_steps = [("×", 4), ("+", 2), ("×", 4)]

def run_chain(steps):
    """Run the given operation chain from 1 and return the final value."""
    value = 1
    for op, x in steps:
        value = value * x if op == "×" else value + x
    return value

print("Final value of the greedy single path (step 2 replaced by +2):", run_chain(bad_steps))
print("Target value:", 40)

# Keep several candidates: at each step retain two ways of proceeding so a wrong
# branch does not crowd out the correct one.
frontier = [[1]]
for i in range(3):
    nxt = []
    for path in frontier:
        for steps in (true_steps, bad_steps):
            op, x = steps[i]
            cur = path[-1]
            nxt.append(path + [cur * x if op == "×" else cur + x])
    frontier = nxt

values = sorted(p[-1] for p in frontier)
print("Final values after keeping two candidates:", values)
print("Number of paths that reach 40:", sum(1 for v in values if v == 40))


The two printed lines contrast the two approaches. A single path stores one trajectory: step 2 used +2, the value stops at 24, and a later correct ×4 still cannot return to 40.

Keeping two candidates expands every old path once under each of the two policies, so the path count grows 1 → 2 → 4 → 8. Steps 1 and 3 happen to be ×4 under both policies; only step 2 differs (+6 versus +2). The eight paths therefore collapse to two final values: the four that took +6 reach 40, and the four that took +2 stop at 24. The sorted output [24, 24, 24, 24, 40, 40, 40, 40] is exactly that result.

The point is retention: even if +2 is seen first at step 2, the +6 branch is not discarded, so the correct final value 40 is still present. Tree search extends this kind of retention to every step and every state.

The previous example showed that walking a single path to the end can stall. This section covers the first response: split the task into smaller pieces. The design choice is when to split, and how finely. ADaPT's answer is: split only when needed.

Decomposition cuts a large task into smaller ones. The most direct method is to write a full plan and then execute it (plan-and-execute): split the task into the smallest steps up front, then run them in order. The limit is that we cannot know in advance which subtasks are hard and which are easy. Forcing a split on a subtask that could have been finished in one step introduces extra actions and extra false assumptions.

ADaPT reverses the order: first let the executor try the whole task; only when the executor reports failure does the planner split it into subtasks, and the same procedure is then called recursively on each subtask. Split depth is set by the true difficulty of the task, not by a depth chosen in advance. The executor reports "I completed it" or "I failed," and that self-report is the success signal for the task.

The two roles stay separate: the planner splits a task into subtasks, the executor actually runs them, and a fixed program, the controller, drives the recursion. Subtasks combine in two ways: AND means every subtask must succeed; OR means any one success is enough.

This subsection builds a minimal environment for comparing three decomposition strategies, so we can see what "split only when needed" actually saves.

Each item has a recipe (the required sub-items). Atomic items need no sub-items. The executor can finish atomic items and one-step items (every sub-item in the recipe is atomic); items that need a deeper combination cause the executor to fail. An environment with a limited executor makes as-needed decomposition necessary.

In [ ]:
# Crafting-recipe environment: item → required sub-items (empty list = atomic)
RECIPES = {
    "stick": [],
    "iron": [],
    "plank": ["stick"],
    "sword": ["plank", "iron"],
    "armory": ["sword", "iron"],
}

def is_atomic(item):
    """Whether the item is atomic: it needs no sub-items."""
    return len(RECIPES[item]) == 0

def is_direct(item):
    """Whether the item is one-step: every sub-item in the recipe is atomic."""
    return all(is_atomic(sub) for sub in RECIPES[item])

def executor_ability(item):
    """Executor completion rule: atomic or one-step items succeed, otherwise fail."""
    if is_atomic(item) or is_direct(item):
        return "completed"
    return "failed"

for item in RECIPES:
    print(f"{item}: recipe={RECIPES[item]}, executor={executor_ability(item)}")


The executor rule is two lines: if the item is atomic, or every sub-item in the recipe is atomic, the verdict is completed; otherwise failed. Checking item by item:

| Item | Recipe | All sub-items atomic | Executor verdict |
|:---|:---|:---|:---|
| stick | [] | yes (no sub-items) | completed |
| iron | [] | yes (no sub-items) | completed |
| plank | [stick] | stick is atomic, yes | completed |
| sword | [plank, iron] | plank is not atomic, no | failed |
| armory | [sword, iron] | sword is not atomic, no | failed |

The executor cannot finish a sword because the plank it depends on has a further dependency (plank → stick), so the combination is deeper than one step. The executor is deliberately limited to one step; that weak executor is what makes decomposition necessary. If the executor could finish every item, planning would not be needed.

In [ ]:
# Locate llm_client.py at the repo root and create a shared client
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm

client = get_llm()
print("Client is a live API demo:", False)

def planner(client, item):
    """Planner: split a task into subtasks and return (subtask list, combination logic).
    Live mode asks the LLM and parses leniently; a live API demo returns the recipe as a scripted plan."""
    if True:
        prompt = f"Decompose 'craft {item}' into subtasks joined by AND, and output a plan of the form A AND B."
        reply = client.chat([{"role": "user", "content": prompt}])
        subs = [s.strip() for s in reply.replace("AND", " and ").split(" and ")]
        subs = [s for s in subs if s in RECIPES]
        if subs:
            return subs, "AND"
    return list(RECIPES[item]), "AND"

def executor(client, item, depth):
    """Executor: return (outcome, self-report text).
    Outcome is decided by the environment's deterministic rule; the self-report comes from the LLM (a scripted example uses a scripted placeholder)."""
    outcome = executor_ability(item)
    if False:
        report = f"scripted placeholder: execute '{item}' → {outcome}"
    else:
        msg = f"Please execute 'craft {item}' and self-report: output completed on success, failed on failure."
        report = client.chat([{"role": "user", "content": msg}])
    return outcome, report

print("planner output:", planner(client, "sword"))
print("executor output:", executor(client, "sword", 0))


In [ ]:
# ADaPT recursive controller: try the executor first; split with the planner only on failure, then recurse
def adapt_controller(client, item, depth, dmax, trace, path):
    """ADaPT(Task, depth): if depth exceeds dmax, run only the executor.
    trace records the call tree as (path, item, action, detail); path is the ancestor sequence."""
    outcome, _ = executor(client, item, depth)
    trace.append((path, item, "executor", outcome))
    if outcome == "completed":
        return True
    if depth >= dmax:
        return False
    subs, logic = planner(client, item)
    trace.append((path, item, "split", list(subs)))
    results = [adapt_controller(client, s, depth + 1, dmax, trace, path + (item,))
               for s in subs]
    return all(results) if logic == "AND" else any(results)

def run_react(client, item):
    """ReAct baseline: run the executor once, with no decomposition."""
    trace = []
    outcome, _ = executor(client, item, 0)
    trace.append(((), item, "executor", outcome))
    return trace, outcome == "completed"

def run_plan_execute(client, item):
    """Plan-and-Execute baseline: split all the way to atomic items, then execute each."""
    trace = []
    def plan_all(it, path):
        if is_atomic(it):
            outcome, _ = executor(client, it, len(path))
            trace.append((path, it, "executor", outcome))
            return
        trace.append((path, it, "split", list(RECIPES[it])))
        for sub in RECIPES[it]:
            plan_all(sub, path + (it,))
    plan_all(item, ())
    return trace, True


In [ ]:
dmax = 3
item = "armory"

react_trace, react_ok = run_react(client, item)
plan_trace, plan_ok = run_plan_execute(client, item)
adapt_trace = []
adapt_ok = adapt_controller(client, item, 0, dmax, adapt_trace, ())

def show_trace(trace):
    """Print the call tree as text."""
    lines = []
    for path, it, action, detail in trace:
        indent = "  " * len(path)
        if action == "split":
            lines.append(f"{indent}{it}: split into {detail}")
        else:
            lines.append(f"{indent}{it}: {detail}")
    return "\n".join(lines)

print("=== ReAct (no decomposition) ===")
print(show_trace(react_trace))
print("Overall success:", react_ok)
print()

print("=== Plan-and-Execute (split everything up front) ===")
print(show_trace(plan_trace))
print("Overall success:", plan_ok)
print()

print("=== ADaPT (split only on failure) ===")
print(show_trace(adapt_trace))
print("Overall success:", adapt_ok)

def count_actions(trace):
    """Count executor and split occurrences."""
    n_exec = sum(1 for _, _, a, _ in trace if a == "executor")
    n_split = sum(1 for _, _, a, _ in trace if a == "split")
    return n_exec, n_split

for name, trace in [("ReAct", react_trace), ("Plan-and-Execute", plan_trace),
                    ("ADaPT", adapt_trace)]:
    ne, ns = count_actions(trace)
    print(f"{name}: executor calls {ne}, splits {ns}")

print("Reading: ADaPT splits only at failures (armory, sword); the plank is finished directly by the executor. "
      "Plan-and-Execute also splits the plank into a stick. One fewer split, and a few more failed executor attempts.")


Match the ADaPT column in the output against the recursion on armory:

1. The executor tries armory directly and returns failed — armory needs a sword, the sword needs a plank, and the executor can only do one step.
2. The planner splits armory into [sword, iron] joined by AND: both must succeed.
3. Recurse on sword: the executor tries again and still fails; the planner splits into [plank, iron].
4. Recurse on plank: the executor finishes in one try (the sub-item stick is atomic); iron likewise finishes in one try.
5. Both subtasks of sword succeed, so AND holds and sword succeeds; iron also succeeds, so armory succeeds.

The whole trace splits only at armory and sword: 5 executor calls and 2 splits. Compare Plan-and-Execute: it also splits plank into [stick], one extra split, even though the executor can finish a plank directly. ReAct does not split, so armory fails immediately.

As-needed shows up at step 4: plank is not forced into a finer split, because the executor can already finish it. Split depth is set by execution results, not written in advance. dmax=3 is a safety bound that stops unbounded recursion.

In [ ]:
import matplotlib.pyplot as plt

# English labels for plot text
EN = {"stick": "stick", "iron": "iron", "plank": "plank",
      "sword": "sword", "armory": "armory"}

def build_tree(trace):
    """Recover node info and parent–child links from a trace."""
    info, children, roots = {}, {}, []
    for path, it, action, detail in trace:
        nid = tuple(list(path) + [it])
        info.setdefault(nid, (it, action, detail))
        children.setdefault(nid, [])
        if path:
            children.setdefault(path, []).append(nid)
        else:
            roots.append(nid)
    return info, children, roots

def layout(nid, children, counter):
    """Bottom-up coordinates: leaves numbered in order, internal nodes at the midpoint of their children."""
    if not children.get(nid):
        x = counter[0]
        counter[0] += 1
        return {nid: x}, x, x
    result, xmin, xmax = {}, 1e9, -1e9
    for c in children.get(nid, []):
        sub, lo, hi = layout(c, children, counter)
        result.update(sub)
        xmin, xmax = min(xmin, lo), max(xmax, hi)
    result[nid] = (xmin + xmax) / 2
    return result, xmin, xmax

def plot_trace(trace, title, ax):
    """Draw the call tree: split nodes as boxes, executor nodes colored by completed/failed."""
    info, children, roots = build_tree(trace)
    pos, counter = {}, [0]
    for root in roots:
        sub, _, _ = layout(root, children, counter)
        pos.update(sub)
    depth = {nid: len(nid) - 1 for nid in info}
    for nid in info:
        for c in children.get(nid, []):
            ax.plot([pos[nid], pos[c]], [-depth[nid], -depth[c]],
                    color="#90a4ae", lw=1, zorder=1)
    for nid, (item, action, detail) in info.items():
        x, y = pos[nid], -depth[nid]
        en = EN.get(item, item)
        if action == "executor":
            color = "#4caf50" if detail == "completed" else "#ef5350"
            ax.scatter(x, y, s=2200, c=color, zorder=3)
            ax.text(x, y, en, ha="center", va="center", fontsize=9, zorder=4)
        else:
            ax.add_patch(plt.Rectangle((x - 0.25, y - 0.18), 0.5, 0.36,
                                       fill=False, edgecolor="#455a64", lw=1.5))
            ax.text(x, y, en, ha="center", va="center", fontsize=9)
    ax.set_xticks([])
    ax.set_ylim(-max(depth.values()) - 0.6, 0.5)
    ax.set_ylabel("depth")
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
plot_trace(react_trace, "ReAct", axes[0])
plot_trace(plan_trace, "Plan-and-Execute", axes[1])
plot_trace(adapt_trace, "ADaPT", axes[2])
plt.tight_layout()
plt.show()


This section asks whether, when one route is wrong, several candidates can be tried together. Tree search is that idea.

Decomposition cuts a task vertically, lowering the difficulty of each step. Tree search expands horizontally: the same state keeps several candidate actions, so a wrong branch can still return to the others. `Monte Carlo tree search` (MCTS) organizes the search as a tree: nodes are states, edges are candidate actions, the root is the initial state, and leaves are states not yet expanded.

LATS lets the same LLM play three roles: the Agent samples candidate actions, a value function evaluates states, and a reflector summarizes lessons from failure. It rests on a premise that LLM tasks are reversible — returning to any historical state means feeding the earlier text back in as input; no world simulator is required. This section first works the two core formulas by hand (UCT selection and value backup), then runs a simplified LATS on the Game of 24.

This subsection works the two search steps by hand: which child to descend into (selection), and how to send a result back to ancestors (backup). We start with selection.

Selection uses the UCT formula to pick among children:

$$ UCT(s) = V(s) + w\sqrt{\frac{\ln N(p)}{N(s)}} $$

V(s) is the child's value, N(s) is the child's visit count, N(p) is the parent's visit count, and w controls exploration strength. The first term favors visited, high-value nodes (exploitation); the second favors rarely visited nodes (exploration): when N(s) is small, $\ln N(p)/N(s)$ is large, so under-explored children are tried first. The logarithm sits in the numerator because, as the parent visit count grows, the need to explore grows only slowly.

We check this on a small hand-built tree. The root has been visited 24 times; four children have these values and visit counts:

| Child | V(s) | N(s) |
|:---|:---|:---|
| s1 | 0.30 | 10 |
| s2 | 0.50 | 5 |
| s3 | 0.10 | 8 |
| s4 | 0.00 | 1 |

With w=1, the exploration term for s4 is about 1.78. Its value is 0, yet the total is still the highest, so it is expanded first. Changing w below shows how the choice switches.

Substitute the four children into the formula to see why s4 is chosen. First the numerator ln(24)≈3.178, then each child's exploration term sqrt(3.178 / N(s)):

| Child | V(s) | N(s) | Exploration sqrt(3.178/N) | UCT total |
|:---|:---|:---|:---|:---|
| s1 | 0.30 | 10 | sqrt(0.318) ≈ 0.564 | 0.864 |
| s2 | 0.50 | 5 | sqrt(0.636) ≈ 0.797 | 1.297 |
| s3 | 0.10 | 8 | sqrt(0.397) ≈ 0.630 | 0.730 |
| s4 | 0.00 | 1 | sqrt(3.178) ≈ 1.783 | 1.783 |

s4 has value 0 and only 1 visit, but its exploration term 1.783 is the largest of the four, so the total 1.783 overtakes s2's 1.297. It has been visited least, so we know least about it, and it is worth one trial.

Note also where ln sits. The numerator is the parent visit count: ln(24)≈3.18, ln(1000)≈6.9. Parent visits growing from 24 to 1000 only a little more than doubles the log. Exploration demand does not rise linearly with parent visits; that is why the formula uses a logarithm.

In [ ]:
import numpy as np

children = {"s1": (0.30, 10), "s2": (0.50, 5), "s3": (0.10, 8), "s4": (0.00, 1)}
N_parent = 24.0

def uct(v, n, w):
    """UCT score: value + exploration term."""
    return v + w * np.sqrt(np.log(N_parent) / n)

for w in [0.1, 1.0, 3.0]:
    scores = {k: uct(*val, w) for k, val in children.items()}
    best = max(scores, key=scores.get)
    fmt = "  ".join(f"{k}={s:.3f}" for k, s in scores.items())
    print(f"w={w}: {fmt}  -> select {best}")


The three values of w show the switch between exploitation and exploration.

At w=0.1, the exploration term is compressed to 0.06–0.18 and barely matters, so the total is driven by value: s2 (highest value 0.50) wins. That is exploitation.
At w=1.0, exploration returns to 0.56–1.78; s4 has the largest exploration term and overtakes the higher-value s2. That is exploration.
At w=3.0, exploration is scaled by another factor of three, and s4 wins by a wide margin at 5.348.

w is the weight on exploration: smaller values favor high-value nodes, larger values prefer unvisited nodes. w is a hyperparameter to tune; w=1 is a common starting point.

One detail worth noting: at w=3.0 the scores of s1 and s3 are both about 1.99; the exploration term already outweighs the difference in value.

In [ ]:
# Value backup: a root→A→leaf path, terminal reward r=1
path = [("root", 10, 0.35), ("A", 4, 0.40), ("leaf", 2, 0.50)]
r = 1.0

updated = []
for name, n_old, v_old in path:
    n_new = n_old + 1
    v_new = (v_old * n_old + r) / n_new
    updated.append((name, n_new, v_new))

print("Incremental mean update (r=1):")
for name, n, v in updated:
    print(f"  {name}: N={n}, V={v:.4f}")
print("The leaf reward is sent back to the root; nodes that are higher-value and more often visited move closer to 1.")


Backup is a one-line formula, worked from the leaf upward. The path is root→A→leaf, reward r=1:

- Leaf: N goes from 2 to 3, V = (0.50×2 + 1) / 3 = 2/3 ≈ 0.667
- A: N goes from 4 to 5, V = (0.40×4 + 1) / 5 = 2.6 / 5 = 0.520
- Root: N goes from 10 to 11, V = (0.35×10 + 1) / 11 = 4.5 / 11 ≈ 0.409

Split the formula: V×N is the total reward this node has received. The update (V×N + r)/(N+1) is the old total plus the new reward, divided by the new visit count — the mean recomputed after each backup. Reward r appears only in this round, but it raises the value of every ancestor on the path; that is what backup means.

Two details. First, update N before V, because the denominator of V uses the new N. Second, the reward lands on the leaf, and the root must still be updated, because the next UCT choice happens at the root: when children's values change, the choice follows. Without value backup, search has no memory of exploitation.

The leaf's old value 0.50 was already above the root and A; after the update the leaf is still highest (0.667). If this path returns a reward many times, the leaf and its ancestors converge toward the true mean of the reward.

The previous subsections worked selection and backup separately. This subsection joins them into a full search loop and runs it on the Game of 24.

The environment is Game of 24: given four numbers, each move takes two of them, applies one of the four arithmetic operations, and puts the result back, until one number remains; equaling 24 yields reward 1. Candidate actions are proposed by the LLM (a live API demo uses a deterministic script); the final verdict comes from the environment rule. Node maintenance, selection, and backup are implemented by us.

The paper's six operations are reduced to a four-step loop: select (UCT) → expand (generate candidate children) → evaluate (score with the environment rule) → backup (incremental mean). On failure we also store a reflection string as semantic memory for the next expansion. We start with the environment and candidate generation.

In [ ]:
import re

class State:
    """Search state: a tuple of numbers."""
    __slots__ = ("nums",)

    def __init__(self, nums):
        self.nums = tuple(nums)

    def __hash__(self):
        return hash(self.nums)

    def __eq__(self, other):
        return self.nums == other.nums

    def __repr__(self):
        return str(list(self.nums))

TARGET = 24.0

def evaluate(state):
    """Environment score: 1 if a single remaining number equals 24, otherwise 0."""
    if len(state.nums) == 1:
        return 1.0 if abs(state.nums[0] - TARGET) < 1e-6 else 0.0
    return 0.0

def apply_expr(state, expr):
    """Apply an expression of the form '8 + 3' to a state: replace the two operands with the result.
    Return None if the expression cannot be used."""
    m = re.match(r"(-?\d+(?:\.\d+)?)\s*([+\-*/])\s*(\d+(?:\.\d+)?)$",
                 expr.strip())
    if not m:
        return None
    a, op, b = float(m.group(1)), m.group(2), float(m.group(3))
    nums = list(state.nums)
    if a not in nums or b not in nums:
        return None
    ia = nums.index(a)
    nums_c = nums[:]
    nums_c[ia] = None
    try:
        ib = nums_c.index(b)
    except ValueError:
        return None
    if op == "+":
        r = a + b
    elif op == "-":
        r = a - b
    elif op == "*":
        r = a * b
    else:
        if abs(b) < 1e-12:
            return None
        r = a / b
    keep = [nums[i] for i in range(len(nums)) if i not in (ia, ib)]
    return State(keep + [r])

def scripted_exprs(state, max_k):
    """scripted placeholder: take the first max_k feasible expressions, ranked by closeness of the result to 24."""
    cands = []
    nums = list(state.nums)
    for i in range(len(nums)):
        for j in range(len(nums)):
            if i == j:
                continue
            for op in ["+", "-", "*", "/"]:
                if op == "/" and abs(nums[j]) < 1e-12:
                    continue
                expr = f"{nums[i]} {op} {nums[j]}"
                ns = apply_expr(state, expr)
                if ns is not None:
                    cands.append((abs(ns.nums[-1] - TARGET), expr))
    cands.sort(key=lambda t: t[0])
    return [e for _, e in cands[:max_k]]

def propose_candidates(client, state, max_k=4):
    """Propose candidate actions for a state; return [(expression, new state)].
    Live mode uses the LLM and parses leniently; a live API demo uses a deterministic script."""
    if False:
        exprs = scripted_exprs(state, max_k)
    else:
        prompt = f"The current numbers are {list(state.nums)}. Write an expression using two of them and one of the four arithmetic operators."
        reply = client.chat([{"role": "user", "content": prompt}])
        exprs = re.findall(r"-?\d+(?:\.\d+)?\s*[+\-*/]\s*\d+(?:\.\d+)?", reply)
        if not exprs:
            exprs = scripted_exprs(state, max_k)
    out = []
    for e in exprs:
        ns = apply_expr(state, e)
        if ns is not None:
            out.append((e, ns))
    return out

print("Initial state:", State([1, 2, 3, 4]))
print("Candidate actions:", [e for e, _ in propose_candidates(client, State([1, 2, 3, 4]))])
print("Evaluate [24]:", evaluate(State([24.0])))


In [ ]:
class Node:
    """Search-tree node: state + visit count + value + children."""

    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent
        self.children = []
        self.visits = 0
        self.value = 0.0

def uct_score(node, parent_visits, w):
    """UCT selection score: value + exploration term. Unvisited children are tried first."""
    if node.visits == 0:
        return float("inf")
    return node.value + w * np.sqrt(np.log(parent_visits) / node.visits)

def select(root, w):
    """From the root, descend along the child with the largest UCT score until a leaf."""
    node = root
    while node.children:
        node = max(node.children, key=lambda c: uct_score(c, node.visits, w))
    return node

def expand(node, client, max_k):
    """Expand: generate candidate children for the current state."""
    for expr, ns in propose_candidates(client, node.state, max_k):
        node.children.append(Node(ns, parent=node))

def backprop(node, reward):
    """Backup: update visit counts and values from leaf to root (incremental mean)."""
    while node is not None:
        node.visits += 1
        node.value = (node.value * (node.visits - 1) + reward) / node.visits
        node = node.parent

def lat_search(client, start_state, iterations, w=1.0, max_k=4,
               reflections=None):
    """Simplified LATS: select → expand → evaluate → backup.
    Return (root, chosen state each round, reward each round, successful terminal or None)."""
    root = Node(start_state)
    chosen, rewards, found = [], [], None
    for it in range(iterations):
        leaf = select(root, w)
        chosen.append(leaf.state)
        reward = evaluate(leaf.state)
        rewards.append(reward)
        if reward > 0.5 and found is None:
            found = leaf
        if reward < 0.5 and len(leaf.state.nums) == 1 and reflections is not None:
            reflections.append(f"Round {it + 1} reflection: final value {leaf.state.nums[0]:.4f} "
                               f"is not 24; try a different combination of operations.")
        expand(leaf, client, max_k)
        backprop(leaf, reward)
    return root, chosen, rewards, found

def count_nodes(root):
    """Count nodes in the search tree."""
    total = 0
    stack = [root]
    while stack:
        n = stack.pop()
        total += 1
        stack.extend(n.children)
    return total


Three functions make one search iteration: select walks down the highest-value children to a leaf, expand generates candidate children for that leaf, and backprop sends the leaf's reward back along the path. Note that uct_score returns +∞ for an unvisited child, so newly generated children are always tried first.

Work a full round by hand. Suppose the current tree is as follows (A and B are children of the root; D and E are children of B):

| Node | N | V |
|:---|:---|:---|
| root | 5 | 0.200 |
| A | 3 | 0.000 |
| B | 2 | 0.500 |
| D | 1 | 1.000 |
| E | 1 | 0.000 |

select starts at the root, parent visits N(root)=5, ln(5)≈1.609:

- A: UCT = 0 + sqrt(1.609/3) ≈ 0.732
- B: UCT = 0.5 + sqrt(1.609/2) ≈ 1.397

B is larger, so we descend. At B we compare its two children, N(B)=2, ln(2)≈0.693:

- D: UCT = 1.0 + sqrt(0.693/1) ≈ 1.833
- E: UCT = 0.0 + sqrt(0.693/1) ≈ 0.833

Select D. Assume D is a terminal state, evaluation yields reward r=1, and backup updates:

- D: N 1→2, V = (1.0×1 + 1) / 2 = 1.000
- B: N 2→3, V = (0.5×2 + 1) / 3 ≈ 0.667
- root: N 5→6, V = (0.2×5 + 1) / 6 ≈ 0.333

The round ends. Value is conducted from leaf toward root: D stays at 1.0, B rises from 0.5 to 0.667, root from 0.2 to 0.333. At the same time B's visit count grows, so next round its exploration term shrinks; if value stops rising, selection will lean toward A. Search rotates between exploitation and exploration in this way.

In [ ]:
def collect_layout(root):
    """Layout helper: return (node list, id-to-parent map)."""
    nodes = []
    stack = [root]
    while stack:
        n = stack.pop()
        nodes.append(n)
        stack.extend(n.children)
    parent = {id(c): n for n in nodes for c in n.children}
    return nodes, parent

def depth_of(n, parent):
    """Compute node depth along the parent chain."""
    d = 0
    while id(n) in parent:
        n = parent[id(n)]
        d += 1
    return d

np.random.seed(42)
reflections = []
root, chosen, rewards, found = lat_search(
    client, State([1, 2, 3, 4]), iterations=12, w=1.0, max_k=4,
    reflections=reflections)

print("Search-tree node count:", count_nodes(root))
print("States chosen in the first 8 rounds:")
for i, s in enumerate(chosen[:8]):
    print(f"  round {i + 1}: {s}")

success_iters = [i + 1 for i, r in enumerate(rewards) if r > 0.5]
print("Rounds that received reward 1:", success_iters)

if found is not None:
    path = []
    node = found
    while node is not None:
        path.append(node.state)
        node = node.parent
    print("Path from root to a solution:", [str(s) for s in reversed(path)])

print("Number of failure reflections:", len(reflections))
if reflections:
    print("Last reflection:", reflections[-1])

print("Visit counts and values along the solution path:")
if found is not None:
    node = found
    while node is not None:
        print(f"  {node.state}: N={node.visits}, V={node.value:.3f}")
        node = node.parent


In a live run, scripted candidate generation ranks actions by how close the result is to 24. Search starts at the root [1,2,3,4], moves among shallow states for the first few rounds (first [1,2,12], then [1,3,8]), reaches [1,24] for the first time at round 6, and then concentrates on that branch until round 22 first receives reward 1.

The path from root to a solution is:

[1,2,3,4] --3×4--> [1,2,12] --2×12--> [1,24] --1×24--> [24]

The last printed block gives visit counts and values along the path. Closer to the leaf, value is higher: the root has N=40, V=0.375; [1,2,12] rises to 0.625; [1,24] to 0.800; the terminal [24] to 1.000. That is backup — each time the solution path is selected it brings back reward 1, and nodes nearer the leaf receive more backups, so their values approach 1. Of the root's 40 visits, only about 15 brought back a reward; the rest were 0, so the root value is only 0.375.

In [ ]:
import matplotlib.pyplot as plt

def plot_search_tree(root, found, title):
    """Draw the search tree: nodes colored by value, solution path circled in red."""
    nodes, parent = collect_layout(root)
    pos, counter = {}, [0]
    def assign(n):
        if not n.children:
            x = counter[0]
            counter[0] += 1
            return {id(n): x}, x, x
        res, xmin, xmax = {}, 1e9, -1e9
        for c in n.children:
            sub, lo, hi = assign(c)
            res.update(sub)
            xmin, xmax = min(xmin, lo), max(xmax, hi)
        res[id(n)] = (xmin + xmax) / 2
        return res, xmin, xmax
    pos, _, _ = assign(root)

    fig, ax = plt.subplots(figsize=(12, 5))
    for n in nodes:
        for c in n.children:
            ax.plot([pos[id(n)], pos[id(c)]],
                    [-depth_of(n, parent), -depth_of(c, parent)],
                    color="#b0bec5", lw=0.8, zorder=1)
    for n in nodes:
        x, y = pos[id(n)], -depth_of(n, parent)
        ax.scatter(x, y, s=900, c=[plt.cm.RdYlGn(n.value)], zorder=3)
        label = ",".join(f"{v:g}" for v in n.state.nums)
        ax.text(x, y, label, ha="center", va="center", fontsize=7, zorder=4)
    node = found
    while node is not None:
        x, y = pos[id(node)], -depth_of(node, parent)
        ax.add_patch(plt.Circle((x, y), 0.3, fill=False, color="#d32f2f",
                                lw=2, zorder=5))
        node = parent.get(id(node))
    ax.set_xticks([])
    ax.set_ylabel("depth")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_search_tree(root, found, "LATS search tree (value-colored)")


The first three sections focused on getting the task right: decomposition lowers difficulty, search keeps several paths. This section focuses on getting it done faster. In the traces of long-reasoning models, many steps are independent of one another — reflection, splitting, trial-and-error, independent sub-computations — and running them in sequence wastes time.

SPRINT interleaves planning and execution: the planner produces a batch of mutually independent subtasks, the executor runs them in parallel and syncs back into the main context, forming a rolling loop of plan → execute → sync → plan again. At training time the original sequential trace is rearranged into structured data: split into steps, judge dependencies, build a DAG, pack by stage. A DAG is a directed acyclic graph used to represent dependencies among steps. The core is the stage-number formula, which decides which steps can share a stage and run in parallel.

This subsection derives the stage-number formula, which decides whether two steps can share a stage and run in parallel. The criterion is whether one step depends on the other for execution.

Give each step a stage number σ. Steps with no parent start at 1; a step with parents takes the maximum over parents of "parent stage + whether that parent executed." The formula is:

$$ \sigma(S_i)=\begin{cases}1, & S_i \text{ has no parent}\\
\max_{S_p\in\mathrm{Parents}(S_i)}\big(\sigma(S_p)+\mathbf{1}[E_p\neq\varnothing]\big), & \text{otherwise} \end{cases} $$

A child is deferred to the next stage only when the parent has an execution phase ($E_p\neq\varnothing$). A plan-only parent (for example "split into 40 and 7") does not create a stage boundary, so the child can join the same stage. We check the numbers on a 6-step reasoning trace.

In [ ]:
# A reasoning trace with dependencies: plan steps and execute steps alternate
steps = {
    "S1 read the problem":      {"has_exec": False, "deps": [], "ptok": 30, "etok": 0},
    "S2 split into 40 and 7":  {"has_exec": False, "deps": ["S1 read the problem"], "ptok": 25, "etok": 0},
    "S3 compute 23×40":      {"has_exec": True,  "deps": ["S2 split into 40 and 7"], "ptok": 10, "etok": 120},
    "S4 compute 23×7":       {"has_exec": True,  "deps": ["S2 split into 40 and 7"], "ptok": 10, "etok": 120},
    "S5 sum": {"has_exec": True, "deps": ["S3 compute 23×40", "S4 compute 23×7"],
                "ptok": 15, "etok": 80},
    "S6 verify with 47×23": {"has_exec": True, "deps": ["S5 sum"], "ptok": 20, "etok": 150},
}

def stage_numbers(steps, plan_is_boundary=False):
    """Compute stage numbers from dependencies.
    If plan_is_boundary=True, treat plan-only steps as stage boundaries as well (no optimization)."""
    out = {}
    def solve(name):
        if name in out:
            return out[name]
        deps = steps[name]["deps"]
        if not deps:
            out[name] = 1
            return 1
        val = 0
        for d in deps:
            inc = 1 if (steps[d]["has_exec"] or plan_is_boundary) else 0
            val = max(val, solve(d) + inc)
        out[name] = val
        return val
    for name in steps:
        solve(name)
    return out

s_opt = stage_numbers(steps)
s_no = stage_numbers(steps, plan_is_boundary=True)
for name in steps:
    print(f"{name}: optimized stage {s_opt[name]}  |  no optimization {s_no[name]}")


Compute the six stage numbers from the formula, one step at a time.

S1 has no parent, so it takes 1. S2 depends on S1; S1 is plan-only (has_exec is False), so the increment is 0:

- S1 read the problem: σ = 1
- S2 split into 40 and 7: σ = 1 + 0 = 1

S3 and S4 both depend on S2. S2 is also plan-only, does not execute, and creates no stage boundary, so both computations can enter stage 1:

- S3 compute 23×40: σ = 1 + 0 = 1
- S4 compute 23×7: σ = 1 + 0 = 1

The executions of S3 and S4 do not depend on each other, so they share a stage and run in parallel. S5 sum depends on the execution results of S3 and S4; both parents have has_exec, so the increment is 1:

- S5 sum: σ = max(1+1, 1+1) = 2

S6 verify depends on the execution result of S5: σ = 2 + 1 = 3.

Compare the no-optimization column: treating S2 as a stage boundary as well pushes S3 and S4 to stage 3, and the whole chain becomes 1,2,3,3,4,5 — five stages that can only run in sequence. After optimization it is 1,1,1,1,2,3, three stages. The entire difference is whether the plan-only steps S1 and S2 create a boundary.

The increment $\mathbf{1}[E_p\neq\varnothing]$ reads: add 1 if the parent has an execution phase. Execution results must be waited for; pure planning need not.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def total_tokens(name):
    """Total tokens of a step (plan + execute)."""
    return steps[name]["ptok"] + steps[name]["etok"]

serial = sum(total_tokens(n) for n in steps)
print("Serial sequential tokens:", serial)

groups = {}
for name, st in s_opt.items():
    groups.setdefault(st, []).append(name)
seq_tok = sum(max(total_tokens(n) for n in g) for g in groups.values())
print("Parallel sequential tokens:", seq_tok)
print(f"Saving of parallel relative to serial: {1 - seq_tok / serial:.1%}")
for st in sorted(groups):
    print(f"  stage {st}: {groups[st]}")

# DAG visualization: nodes colored by stage; plot text in English
EN_STEPS = {"S1 read the problem": "read", "S2 split into 40 and 7": "split",
            "S3 compute 23×40": "calc 23x40", "S4 compute 23×7": "calc 23x7",
            "S5 sum": "sum", "S6 verify with 47×23": "verify"}

g = nx.DiGraph()
for name in steps:
    g.add_node(name)
    for d in steps[name]["deps"]:
        g.add_edge(d, name)
pos = nx.spring_layout(g, seed=7, k=1.4)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
colors = [plt.cm.tab10((s_opt[n] - 1) % 10) for n in g.nodes]
nx.draw_networkx_edges(g, pos, ax=axes[0], arrows=True, arrowstyle="-|>",
                       edge_color="#90a4ae")
nx.draw_networkx_nodes(g, pos, ax=axes[0], node_color=colors, node_size=2400)
nx.draw_networkx_labels(g, pos, ax=axes[0], labels=EN_STEPS, font_size=9)
axes[0].set_title("DAG colored by stage")
axes[0].axis("off")

axes[1].bar(["serial", "parallel"], [serial, seq_tok],
            color=["#90a4ae", "#4caf50"])
axes[1].set_title("Sequential tokens")
axes[1].set_ylabel("tokens")
for i, v in enumerate([serial, seq_tok]):
    axes[1].text(i, v + 5, str(v), ha="center")

plt.tight_layout()
plt.show()


580 is the total token count of the six steps run in sequence, added directly:

30 + 25 + 130 + 130 + 95 + 170 = 580

In parallel, steps in the same stage run together, and that stage's time is set by its longest step, so each stage takes only the maximum:

- Stage 1: max(30, 25, 130, 130) = 130
- Stage 2: max(95) = 95
- Stage 3: max(170) = 170

130 + 95 + 170 = 395, a saving of (580 − 395) / 580 ≈ 31.9% relative to 580.

The reason for taking the per-stage maximum: parallelism does not add the times of the steps; it lets the slowest step in the stage set that stage's duration. What is saved is time spent waiting for earlier steps to finish.

## 5. Whether compute should widen or deepen

Earlier sections each implied a hyperparameter: how many candidates to expand at once, or how deep to search. Standard MCTS has a fixed branching width, while repeated sampling only widens (generating new answers from scratch) and sequential refinement only deepens (improving an existing answer). If the task sits between the two, a fixed policy wastes budget.

Wider or Deeper addresses that choice: there is no cap on branching, and each node dynamically decides whether to widen (generate a fresh candidate from the current node, the GEN action) or deepen (refine an existing answer). The selection policy is Thompson sampling rather than UCT — UCT assumes a fixed set of candidate branches, while GEN keeps creating new ones. Thompson sampling draws one number from each action's success-rate posterior; a new branch enters the competition with its prior.


This subsection decides, at each step, whether to widen or deepen. Thompson sampling gives an online rule that adapts the choice to each action's history.

Treat each action (GEN and REFINE) as an arm with unknown success rate, and describe that rate with a Beta(α, β) posterior. Start from a uniform prior α=β=1; after each execution update by the outcome: success α+1, failure β+1. At choice time, draw one success rate from each arm's posterior and take the maximum. That process — random perturbation while exploring, a bias toward the better arm while exploiting — is Thompson sampling.

We run this mini-algorithm on two synthetic arms and watch its preference for the higher-success-rate REFINE.

We do not know an action's true success rate; we only observe some successes and some failures. Using a single number (for example "2 successes and 1 failure, so the rate is 0.67") throws away information — with few observations we are not sure. A distribution over the unknown rate both names the most likely value and expresses how certain we are. Beta(α, β) is such a distribution, defined on (0,1): α can be read as successes plus 1, β as failures plus 1, and the mean is α/(α+β).

Initially α=β=1, a uniform prior: any success rate is equally likely, mean 0.5. Each action updates once:

| History | Posterior | Mean |
|:---|:---|:---|
| nothing yet | Beta(1,1) | 0.500 |
| 1 success | Beta(2,1) | 0.667 |
| 1 success, 1 failure | Beta(2,2) | 0.500 |
| 2 successes, 1 failure | Beta(3,2) | 0.600 |

At choice time we draw one number from each arm's posterior and take the maximum. Sampling noise supplies exploration: even if REFINE has a higher mean, a low draw can still occur, and then GEN can be selected; the higher the mean, the more often a high draw occurs, so exploitation also grows.

UCT is not used here because UCT assumes a fixed set of candidate children. The GEN action keeps creating new branches, so the number of arms changes and N(s) in UCT loses its meaning. Thompson samples only from the current set of arms, and therefore adapts to a changing branch count.

In [ ]:
import numpy as np

class BetaArm:
    """Posterior of one action: Beta(alpha, beta), describing an unknown success rate."""

    def __init__(self, alpha=1, beta=1):
        self.alpha = alpha
        self.beta = beta

    def sample(self, rng):
        """Draw one success rate from the posterior."""
        return rng.beta(self.alpha, self.beta)

    def update(self, success):
        """Update posterior parameters from an observation."""
        self.alpha += success
        self.beta += 1 - success

def scripted_scorer(action, p_gen, p_refine, rng):
    """Synthetic environment: GEN and REFINE each have a true success probability; return 0/1."""
    p = p_gen if action == "gen" else p_refine
    return 1 if rng.random() < p else 0

def thompson_choose(arms, rng):
    """Sample from each arm's posterior and return the action name with the largest success rate."""
    return max(arms, key=lambda a: arms[a].sample(rng))

# Two arms: gen generates a new answer (widen), refine refines an existing answer (deepen)
arms = {"gen": BetaArm(1, 1), "refine": BetaArm(1, 1)}
rng = np.random.default_rng(0)
p_gen, p_refine = 0.20, 0.45

history = []
for _ in range(80):
    action = thompson_choose(arms, rng)
    reward = scripted_scorer(action, p_gen, p_refine, rng)
    arms[action].update(reward)
    history.append((action, reward))

n_gen = sum(1 for a, _ in history if a == "gen")
n_ref = len(history) - n_gen
print("GEN count in 80 rounds:", n_gen, "| REFINE count:", n_ref)
print("GEN posterior Beta:", (arms["gen"].alpha, arms["gen"].beta))
print("REFINE posterior Beta:", (arms["refine"].alpha, arms["refine"].beta))


After 80 rounds, REFINE was selected 69 times and GEN only 11. The two posteriors explain the allocation:

- GEN: Beta(1, 12), all 11 trials failed, mean 1/13 ≈ 0.08
- REFINE: Beta(23, 48), 22 successes in 69 trials, mean 23/71 ≈ 0.32

REFINE's true success rate 0.45 is higher than GEN's 0.20; the posterior mean (0.32) is also clearly above GEN (0.08), so sampled success rates are often higher than GEN's, and REFINE is chosen more often. GEN is not abandoned entirely, because the posterior still has width and sampling sometimes draws a higher value; those 11 trials are the cost of exploration.

Imagine a concrete task where refining an existing answer is more effective than generating a new one. Thompson sampling learns on its own during the run to lean the budget toward REFINE; no one has to specify in advance whether to widen or deepen.

In [ ]:
import matplotlib.pyplot as plt

def simulate(strategy, p_gen, p_refine, budget, seeds=400):
    """Simulate a branching strategy; return the cumulative success-rate curve as the action count grows."""
    curves = np.zeros((seeds, budget))
    for seed in range(seeds):
        rng = np.random.default_rng(seed)
        best = 0.0
        if strategy == "widen":
            for t in range(budget):
                if rng.random() < p_gen:
                    best = 1.0
                curves[seed, t] = best
        elif strategy == "deepen":
            for t in range(budget):
                p = p_gen if t == 0 else p_refine
                if rng.random() < p:
                    best = 1.0
                curves[seed, t] = best
        else:  # adaptive
            arms = {"gen": BetaArm(1, 1), "refine": BetaArm(1, 1)}
            for t in range(budget):
                action = thompson_choose(arms, rng)
                reward = scripted_scorer(action, p_gen, p_refine, rng)
                arms[action].update(reward)
                if reward == 1:
                    best = 1.0
                curves[seed, t] = best
    return curves.mean(axis=0)

budget = 25
p_gen, p_refine = 0.20, 0.45
strategies = {"widen": "Widen (repeated sampling)",
              "deepen": "Deepen (sequential refinement)",
              "adaptive": "Adaptive (Thompson)"}
xs = np.arange(1, budget + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for name, label in strategies.items():
    curve = simulate(name, p_gen, p_refine, budget)
    axes[0].plot(xs, curve, marker="o", markersize=3, label=label)
axes[0].set_xlabel("actions")
axes[0].set_ylabel("success rate")
axes[0].set_title("Widen vs Deepen vs Adaptive")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Action allocation of the adaptive strategy
arms = {"gen": BetaArm(1, 1), "refine": BetaArm(1, 1)}
rng = np.random.default_rng(1)
allocation = []
for _ in range(budget):
    action = thompson_choose(arms, rng)
    reward = scripted_scorer(action, p_gen, p_refine, rng)
    arms[action].update(reward)
    allocation.append(action)

axes[1].bar(["GEN", "REFINE"], [allocation.count("gen"), allocation.count("refine")],
            color=["#90a4ae", "#4caf50"])
axes[1].set_title("Adaptive action allocation")
axes[1].set_ylabel("count")
plt.tight_layout()
plt.show()

print("GEN / REFINE allocation in one run:", allocation.count("gen"),
      "/", allocation.count("refine"))


The three curves on the left are cumulative success rate (average of 400 random traces). widen has hit probability only p_gen=0.20 at each step, so the curve rises slowest, about 0.67 at step 5; deepen raises the hit probability to p_refine=0.45 after the first step, about 0.93 at step 5, clearly faster. With enough budget both approach 1; the difference is the speed of the rise.

adaptive takes the middle path: of a 25-step budget it assigns 17 to REFINE and 8 to GEN (right plot). The curve tracks deepen, without a strategy specified in advance. The simulation sets p_refine larger than p_gen by construction; adaptive learns that preference from the posterior during the run. If p_gen were larger, it would lean toward GEN instead.

## 6. Turning planning experience into training data

The previous four papers all spend compute at inference time (test-time) — decomposition, search, parallelism. SWiRL takes another route: train "when to decompose, when to call a tool, when to stop" directly into the model parameters, so inference no longer searches. It addresses a pain the earlier methods cannot avoid: in a multi-step task one wrong intermediate step cascades into the ending, while ordinary RLHF (reinforcement learning from human feedback) scores only the final answer and cannot tell which step went wrong.

Step one: build multi-step traces from synthetic data. SWiRL lets a seed model run a batch of multi-step tasks in a tool-using environment (search, compute, summarize) and records the full trace "question → several actions → result." No one has to label each step; the trace itself is the training material.

Step two: filter traces with process and outcome signals. The traces that come back vary in quality, and SWiRL screens them with two signals. Outcome filtering keeps only traces whose final answer is correct — coarse, and it cannot tell which step guessed right. Process filtering judges each step on its own: whether that tool call was necessary, whether the retrieval was relevant, and keeps traces with high step-level quality. The verifier idea from Lecture 3 appears again: the verifier moves from "picking an answer" to "picking a trace, picking a step."

Step three: step-wise RL. Cut a multi-step trace into prefix sub-traces at each action, and run RL with a generative reward model scoring each step. Compared with outcome RL, which gives one reward at the end, step-wise RL turns a sparse final signal into a dense per-step signal, and can tell how much each step contributed to the final result — that is `credit assignment`. It is the same idea as Lecture 6's GRPO of "assigning reward to each step."

Relation to the previous four papers. LATS searches actions at inference time, ADaPT decomposes tasks at inference time; both must be re-run every time, at high cost. SWiRL distills those "when to split, when to call a tool" policies into the weights, so one forward pass at inference time is enough. The paper reports gains of about 21.5% (four-step tasks) and 12.3% (two-step tasks) over the baseline on multi-step tool tasks. It shows that search-style policies can also be trained into the model, and this lecture therefore leads into Lecture 6's train-time scaling.

## Summary

This lecture is organized around which path to take next in a multi-step task. What we covered:

- [ ] Limits of single-step reasoning: holistic generation and greedy decoding both let errors accumulate along a single path, with no way back
- [ ] ADaPT as-needed decomposition: the executor tries first, the planner splits only on failure, recursion is controlled, AND/OR combination; split depth is set by task difficulty
- [ ] LATS tree search: nodes are states, UCT balances exploration and exploitation, value backup uses an incremental mean, reflection is semantic memory
- [ ] SPRINT parallelism: the stage-number formula packs independent steps; a plan-only parent does not create a stage boundary
- [ ] Wider or Deeper adaptive branching: GEN widens, REFINE deepens, Thompson sampling decides the allocation online
- [ ] Decomposition addresses task difficulty, search addresses path uncertainty, parallelism addresses latency, training addresses the cost of searching from scratch every time

There is also a training-side route: SWiRL does not rely on inference-time search, and instead trains "when to decompose, when to call a tool" directly into the model parameters. The methods in this lecture all consume inference-time compute; SWiRL trains planning into the model, and is the bridge into Lecture 6's train-time scaling.

## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.

Each of the three problems has one blank. Reference answers are already filled into the code. Work the calculation by hand first, then run the cell and check the asserts.


**Exercise 1: complete the UCT selection formula**

Given four children of a tree (each with a value and a visit count) and the parent visit count, complete the UCT formula and select the child that should be expanded at w=1.

Hint: the exploration term is $w\sqrt{\ln N(p)/N(s)}$. Decide which quantity grows in a way that makes exploration demand increase only slowly.


In [ ]:
import numpy as np

# Exercise 1: complete the UCT selection formula
# The blank should be filled with w * np.sqrt(np.log(N_parent) / n)
children_v = np.array([0.30, 0.50, 0.10, 0.00])
children_n = np.array([10, 5, 8, 1])
N_parent = 24.0
w = 1.0

def uct_score(v, n):
    return v + w * np.sqrt(np.log(N_parent) / n)   # blank is here

scores = uct_score(children_v, children_n)
print("UCT scores:", np.round(scores, 3))
assert int(np.argmax(scores)) == 3, "selected child does not match the hand calculation"
print("Index of the selected child:", int(np.argmax(scores)))
print("Takeaway: with w fixed, a child with fewer visits scores higher because of the exploration term")


**Exercise 2: complete value backup (incremental mean)**

Given a root→A→leaf path and each node's old visit count and old value, with reward r=1, complete the backup update and check the new values of the leaf and the root.

Hint: $V(s)\leftarrow\big(V(s)\cdot N(s)+r\big)/(N(s)+1)$. Compute the new N first, then the new V.


In [ ]:
# Exercise 2: complete LATS value backup (incremental mean)
# The blank should be filled with (v_old * n_old + r) / n_new
path = [("root", 10, 0.35), ("A", 4, 0.40), ("leaf", 2, 0.50)]
r = 1.0

updated = []
for name, n_old, v_old in path:
    n_new = n_old + 1
    v_new = (v_old * n_old + r) / n_new   # blank is here
    updated.append((name, n_new, v_new))

for name, n, v in updated:
    print(f"{name}: N={n}, V={v:.4f}")

assert abs(updated[-1][2] - (0.50 * 2 + 1) / 3) < 1e-6, "leaf value backup does not match"
assert abs(updated[0][2] - (0.35 * 10 + 1) / 11) < 1e-6, "root value backup does not match"
print("Takeaway: reward r is sent back layer by layer along the path from root to leaf; update the visit count first, then the value")


**Exercise 3: complete SPRINT stage-number computation**

Given a dependency table that includes a plan-only parent, complete the increment term in the stage-number formula, and check that two independent computations share a stage.

Hint: defer a child to the next stage only when the parent has an execution phase (has_exec is True).


In [ ]:
# Exercise 3: complete SPRINT stage-number computation (including the plan-only optimization)
# The blank should be filled with 1 if steps[d]["has_exec"] else 0
steps = {
    "S1 read the problem":     {"has_exec": False, "deps": []},
    "S2 split into 40 and 7": {"has_exec": False, "deps": ["S1 read the problem"]},
    "S3 compute 23×40":     {"has_exec": True,  "deps": ["S2 split into 40 and 7"]},
    "S4 compute 23×7":      {"has_exec": True,  "deps": ["S2 split into 40 and 7"]},
    "S5 sum":         {"has_exec": True,  "deps": ["S3 compute 23×40", "S4 compute 23×7"]},
}

def stage_numbers(steps):
    out = {}
    def solve(name):
        if name in out:
            return out[name]
        deps = steps[name]["deps"]
        if not deps:
            out[name] = 1
            return 1
        val = 0
        for d in deps:
            inc = 1 if steps[d]["has_exec"] else 0   # blank is here
            val = max(val, solve(d) + inc)
        out[name] = val
        return val
    for name in steps:
        solve(name)
    return out

s = stage_numbers(steps)
for name, v in s.items():
    print(f"{name}: stage {v}")

assert s["S3 compute 23×40"] == 1 and s["S4 compute 23×7"] == 1, "the two independent computations should share stage 1"
assert s["S5 sum"] == 2, "sum depends on execution results and should lag by one stage"
print("Takeaway: a plan-only parent does not push children into the next stage, so independent steps can run in parallel")


## References

- Zhou et al., [Language Agent Tree Search Unifies Reasoning, Acting, and Planning in Language Models](https://arxiv.org/abs/2310.04406) — main tree-search thread of this lecture; the first general framework combining MCTS with an LLM
- Prasad et al., [ADaPT: As-Needed Decomposition and Planning with Language Models](https://arxiv.org/abs/2311.05772) — decomposition: as-needed recursive splits and AND/OR plans
- Biju et al., [SPRINT: Enabling Interleaved Planning and Parallelized Execution in Reasoning Models](https://arxiv.org/abs/2506.05745) — parallel execution, DAG packing, and the planner/executor rolling loop
- Inoue et al., [Wider or Deeper? Scaling LLM Inference-Time Compute with Adaptive Branching Tree Search](https://arxiv.org/abs/2503.04412) — adaptive branching, GEN nodes, and Thompson sampling
- Goldie et al., [SWiRL: Synthetic Data Generation & Multi-Step RL for Reasoning & Tool Use](https://arxiv.org/abs/2504.04736) — training view: step-wise RL and process/outcome filtering
- Prior concepts (optional review): Chain-of-Thought (Wei et al., 2022), Self-Consistency (Wang et al., 2022), ReAct (Yao et al., 2023), Tree-of-Thought (Yao et al., 2023), Reflexion (Shinn et al., 2023), UCT (Kocsis & Szepesvári, 2006)
